# Building master_dataset.csv from 8 Source Tables

This notebook walks through the complete, step-by-step construction of `master_dataset.csv`
by loading, enriching, aggregating and joining all **8 source tables**.

| # | Source Table | Grain | Role |
|---|---|---|---|
| 1 | `customers.csv` | 1 row per customer | Dimension — customer profile & segment |
| 2 | `products.csv` | 1 row per product | Dimension — product catalog & margins |
| 3 | `orders.csv` | 1 row per order | Fact — revenue, channel, return status |
| 4 | `order_items.csv` | 1 row per line item | Fact — item-level revenue & quantity |
| 5 | `inventory_daily.csv` | 1 row per product per day | Fact — stock levels & stockouts |
| 6 | `marketing_spend_daily.csv` | 1 row per channel per day | Fact — ad spend & ROAS by channel |
| 7 | `website_traffic_daily.csv` | 1 row per day | Fact — sessions, bounce, conversion |
| 8 | `anomaly_log.csv` | 1 row per anomaly event | Label — ground-truth anomaly flags |

**Pipeline:**
```
customers + products  →  enrich orders & order_items (dimension joins)
orders                →  aggregate to daily KPIs
marketing_spend       →  aggregate 5 channels to daily totals
inventory_daily       →  aggregate 500 products to daily totals
website_traffic       →  already daily (rename one column)
anomaly_log           →  left-join on date (fill nulls to 0/'')
latent drivers        →  left-join from master_dataset
                      ────────────────────────────────
                      master_dataset.csv  731 rows × 33 cols
```

---

## 0 · Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
pd.set_option('display.width', 120)

DATA = Path('..') / 'data'
print('Data directory:', DATA.resolve())

Data directory: C:\Users\annes\OneDrive\Adidas Office One Drive - Personal Folders\Upskilling\2026\KPI Anomaly Detection Agent\data


---
## 1 · Load All 8 Source Tables

Each table is loaded and inspected with `.head()` and `.shape`.

### 1.1 · customers.csv — Customer Dimension

In [2]:
customers = pd.read_csv(DATA / 'customers.csv')
print(f'Shape: {customers.shape}  |  Segments: {sorted(customers["segment"].unique())}')
print(f'Countries: {sorted(customers["country"].unique())}')
customers.head()

Shape: (10000, 9)  |  Segments: ['churned', 'loyalty', 'new', 'occasional', 'regular']
Countries: ['AE', 'AU', 'CA', 'DE', 'FR', 'JP', 'UK', 'US']


,customer_id,segment,country,cohort_month,age,is_loyalty_member,lifetime_value_usd,email_opt_in,avg_review_score
0,C0000001,new,FR,2024-09,23,0,84.5700,0,3.6000
1,C0000002,regular,FR,2024-03,72,0,228.5400,0,1.5000
2,C0000003,new,US,2022-08,47,0,87.9000,1,3.6000
3,C0000004,occasional,US,2024-12,74,0,60.9800,0,3.6000
4,C0000005,loyalty,US,2024-12,41,1,779.1300,0,2.9000


### 1.2 · products.csv — Product Dimension

In [3]:
products = pd.read_csv(DATA / 'products.csv')
print(f'Shape: {products.shape}  |  Categories: {sorted(products["category"].unique())}')
products.head()

Shape: (500, 7)  |  Categories: ['Apparel', 'Beauty', 'Books', 'Electronics', 'Food', 'Home', 'Sports', 'Toys']


,product_id,category,brand,base_price_usd,cost_usd,gross_margin,is_premium
0,P000001,Toys,Brand_F,186.0400,93.7300,0.4960,0
1,P000002,Books,Brand_G,52.0000,23.0000,0.5580,0
2,P000003,Home,Brand_N,470.1900,200.7900,0.5730,1
3,P000004,Apparel,Brand_K,269.7400,114.7100,0.5750,0
4,P000005,Beauty,Brand_F,123.7900,59.8000,0.5170,0


### 1.3 · orders.csv — Order Transactions

In [4]:
orders = pd.read_csv(DATA / 'orders.csv', parse_dates=['order_date'])
print(f'Shape: {orders.shape}  |  Date range: {orders["order_date"].min().date()} to {orders["order_date"].max().date()}')
print(f'Channels: {sorted(orders["channel"].unique())}')
print(f'Statuses: {sorted(orders["status"].unique())}')
orders.head()

Shape: (148084, 8)  |  Date range: 2024-01-01 to 2025-12-31
Channels: ['affiliate', 'direct', 'email', 'organic_search', 'paid_search', 'referral', 'social']
Statuses: ['completed', 'returned']


,order_id,customer_id,order_date,channel,status,order_total_usd,discount_pct,country
0,O000000001,C0008742,2024-01-01,email,completed,129.9100,0.0700,US
1,O000000002,C0007738,2024-01-01,organic_search,completed,14.8600,0.0700,FR
2,O000000003,C0000204,2024-01-01,direct,completed,73.9200,0.0700,DE
3,O000000004,C0005038,2024-01-01,organic_search,completed,6.2800,0.0700,US
4,O000000005,C0003886,2024-01-01,organic_search,completed,80.8100,0.0700,UK


### 1.4 · order_items.csv — Line Items

In [5]:
order_items = pd.read_csv(DATA / 'order_items.csv')
print(f'Shape: {order_items.shape}  |  Unique orders: {order_items["order_id"].nunique():,}  |  Unique products: {order_items["product_id"].nunique()}')
order_items.head()

Shape: (370081, 7)  |  Unique orders: 148,084  |  Unique products: 500


,order_item_id,order_id,product_id,quantity,unit_price_usd,discount_pct,line_total_usd
0,OI0000000001,O000000001,P000255,1,129.9100,0.0700,129.9100
1,OI0000000002,O000000002,P000066,2,7.1100,0.0700,14.2200
2,OI0000000003,O000000002,P000160,2,0.3200,0.0700,0.6400
3,OI0000000004,O000000003,P000397,1,44.5400,0.0700,44.5400
4,OI0000000005,O000000003,P000066,2,0.7400,0.0700,1.4800


### 1.5 · inventory_daily.csv — Daily Stock per Product

In [6]:
inventory = pd.read_csv(DATA / 'inventory_daily.csv', parse_dates=['date'])
print(f'Shape: {inventory.shape}  |  Products: {inventory["product_id"].nunique()}  |  Days: {inventory["date"].nunique()}')
inventory.head()

Shape: (365500, 6)  |  Products: 500  |  Days: 731


,date,product_id,stock_on_hand,units_sold,reorder_triggered,stockout_flag
0,2024-01-01,P000001,13,0,1,0
1,2024-01-01,P000002,172,0,0,0
2,2024-01-01,P000003,27,0,0,0
3,2024-01-01,P000004,50,1,0,0
4,2024-01-01,P000005,306,1,0,0


### 1.6 · marketing_spend_daily.csv — Daily Spend per Channel

In [7]:
marketing = pd.read_csv(DATA / 'marketing_spend_daily.csv', parse_dates=['date'])
print(f'Shape: {marketing.shape}  |  Channels: {sorted(marketing["channel"].unique())}')
marketing.head()

Shape: (3655, 8)  |  Channels: ['affiliate', 'display', 'email', 'paid_search', 'social']


,date,channel,spend_usd,impressions,clicks,conversions,attributed_revenue_usd,roas
0,2024-01-01,paid_search,4037.9100,739065,28006,823,161564.6600,40.0120
1,2024-01-01,social,3590.0600,657093,24900,732,143645.1300,40.0120
2,2024-01-01,email,2068.5300,378606,14347,422,82765.9500,40.0120
3,2024-01-01,affiliate,1038.5400,190085,7203,211,41553.9500,40.0120
4,2024-01-01,display,1414.6000,258920,9813,291,56601.1600,40.0120


### 1.7 · website_traffic_daily.csv — Daily Web Traffic

In [8]:
traffic = pd.read_csv(DATA / 'website_traffic_daily.csv', parse_dates=['date'])
print(f'Shape: {traffic.shape}  |  Already at daily grain — one row per day')
traffic.head()

Shape: (731, 8)  |  Already at daily grain — one row per day


,date,sessions,unique_visitors,bounce_rate,pages_per_session,avg_session_duration_sec,conversion_rate,conversions
0,2024-01-01,7706,6486,0.4110,3.1900,162.9000,0.0250,192
1,2024-01-02,7759,6552,0.4110,3.3900,189.3000,0.0246,190
2,2024-01-03,7739,5958,0.4020,3.2900,183.5000,0.0245,189
3,2024-01-04,7723,5733,0.4740,2.9500,199.9000,0.0223,172
4,2024-01-05,7689,5882,0.4260,3.3900,170.6000,0.0243,186


### 1.8 · anomaly_log.csv — Ground-Truth Anomaly Labels

In [9]:
anomaly_log = pd.read_csv(DATA / 'anomaly_log.csv', parse_dates=['date'])
print(f'Shape: {anomaly_log.shape}  |  {len(anomaly_log)} labeled anomaly events across the 2-year dataset')
anomaly_log

Shape: (20, 3)  |  20 labeled anomaly events across the 2-year dataset


,date,anomaly_event,anomaly_kpi
0,2024-02-08,economic_sentiment_drop,aov
1,2024-03-15,inventory_stockout,sales_volume
2,2024-04-22,logistics_disruption,fulfillment_time
3,2024-05-22,fraud_attack,refunds
4,2024-06-18,website_outage,sessions
5,2024-07-10,marketing_tracking_outage,roas
6,2024-08-05,defective_product_returns,return_rate
7,2024-08-20,back_to_school_surge,sales_volume
8,2024-09-03,email_campaign_spike,conversion_rate
9,2024-10-14,coupon_abuse,discount_rate


---
## 2 · Dimension Joins (Enrichment)

`customers.csv` and `products.csv` are **dimension tables** — they don't contribute columns
directly to `master_dataset.csv`, but they enrich the fact tables for segment- and
category-level analytics.

### 2.1 · orders + customers  →  Customer Segment Enrichment

In [10]:
# Join customer segment and loyalty flag onto each order
orders_enriched = orders.merge(
    customers[['customer_id', 'segment', 'is_loyalty_member']],
    on='customer_id',
    how='left'
)

print(f'orders_enriched shape: {orders_enriched.shape}')
print(f'FK coverage — all orders matched a customer: {orders_enriched["segment"].notna().all()}')
orders_enriched.head()

orders_enriched shape: (148084, 10)


FK coverage — all orders matched a customer: True


,order_id,customer_id,order_date,channel,status,order_total_usd,discount_pct,country,segment,is_loyalty_member
0,O000000001,C0008742,2024-01-01,email,completed,129.9100,0.0700,US,occasional,0
1,O000000002,C0007738,2024-01-01,organic_search,completed,14.8600,0.0700,FR,regular,0
2,O000000003,C0000204,2024-01-01,direct,completed,73.9200,0.0700,DE,loyalty,1
3,O000000004,C0005038,2024-01-01,organic_search,completed,6.2800,0.0700,US,regular,0
4,O000000005,C0003886,2024-01-01,organic_search,completed,80.8100,0.0700,UK,loyalty,1


In [11]:
# Example: revenue by customer segment (analytical use of dimension join)
seg_revenue = (
    orders_enriched
    .groupby('segment')['order_total_usd']
    .agg(total_revenue='sum', order_count='count', avg_order='mean')
    .sort_values('total_revenue', ascending=False)
    .round(2)
)
print('Revenue breakdown by customer segment:')
seg_revenue

Revenue breakdown by customer segment:


,total_revenue,order_count,avg_order
segment,,,
regular,2606230.9500,45268,57.5700
occasional,2136982.6300,36944,57.8400
new,1670877.1200,28846,57.9200
loyalty,1292551.6700,22226,58.1500
churned,857341.1900,14800,57.9300


### 2.2 · order_items + products  →  Product Category Enrichment

In [12]:
# Join product category, brand and gross margin onto each line item
items_enriched = order_items.merge(
    products[['product_id', 'category', 'brand', 'gross_margin']],
    on='product_id',
    how='left'
)

print(f'items_enriched shape: {items_enriched.shape}')
print(f'FK coverage — all items matched a product: {items_enriched["category"].notna().all()}')
items_enriched.head()

items_enriched shape: (370081, 10)
FK coverage — all items matched a product: True


,order_item_id,order_id,product_id,quantity,unit_price_usd,discount_pct,line_total_usd,category,brand,gross_margin
0,OI0000000001,O000000001,P000255,1,129.9100,0.0700,129.9100,Sports,Brand_F,0.5430
1,OI0000000002,O000000002,P000066,2,7.1100,0.0700,14.2200,Food,Brand_N,0.4450
2,OI0000000003,O000000002,P000160,2,0.3200,0.0700,0.6400,Toys,Brand_N,0.5400
3,OI0000000004,O000000003,P000397,1,44.5400,0.0700,44.5400,Electronics,Brand_L,0.6100
4,OI0000000005,O000000003,P000066,2,0.7400,0.0700,1.4800,Food,Brand_N,0.4450


In [13]:
# Example: revenue by product category
cat_revenue = (
    items_enriched
    .groupby('category')['line_total_usd']
    .agg(total_revenue='sum', items_sold='count', avg_margin=lambda x: items_enriched.loc[x.index, 'gross_margin'].mean())
    .sort_values('total_revenue', ascending=False)
    .round(2)
)
print('Revenue breakdown by product category:')
cat_revenue

Revenue breakdown by product category:


,total_revenue,items_sold,avg_margin
category,,,
Apparel,1587694.9300,68650,0.4900
Home,1381763.0100,59704,0.4900
Electronics,1318989.7300,56857,0.5000
Toys,1072499.5200,46513,0.5100
Sports,988583.6500,42946,0.4800
Food,814827.0400,35400,0.5100
Books,746959.3800,31847,0.5000
Beauty,652671.7600,28164,0.5000


---
## 3 · Aggregate Fact Tables to Daily Grain

Each fact table is aggregated so it has **one row per calendar day** before merging.

### 3.1 · orders.csv  →  Daily Order KPIs

In [14]:
# Tag return flag on each order
orders['is_return'] = (orders['status'] == 'returned').astype(int)

orders_daily = (
    orders
    .groupby('order_date')
    .agg(
        n_orders            = ('order_id',        'count'),
        total_revenue_usd   = ('order_total_usd', 'sum'),
        avg_order_value_usd = ('order_total_usd', 'mean'),
        avg_discount_pct    = ('discount_pct',    'mean'),
        n_unique_customers  = ('customer_id',     'nunique'),
        n_returns           = ('is_return',       'sum'),
    )
    .reset_index()
    .rename(columns={'order_date': 'date'})
)

orders_daily['return_rate']          = (orders_daily['n_returns'] / orders_daily['n_orders']).round(4)
orders_daily['total_revenue_usd']    = orders_daily['total_revenue_usd'].round(2)
orders_daily['avg_order_value_usd']  = orders_daily['avg_order_value_usd'].round(2)
orders_daily['avg_discount_pct']     = orders_daily['avg_discount_pct'].round(4)

print(f'orders_daily shape: {orders_daily.shape}  (one row per day, aggregated from {len(orders):,} individual orders)')
orders_daily.head()

orders_daily shape: (731, 8)  (one row per day, aggregated from 148,084 individual orders)


,date,n_orders,total_revenue_usd,avg_order_value_usd,avg_discount_pct,n_unique_customers,n_returns,return_rate
0,2024-01-01,194,10670.0000,55.0000,0.0700,153,15,0.0773
1,2024-01-02,197,11219.0000,56.9500,0.0700,158,15,0.0761
2,2024-01-03,195,10950.5400,56.1600,0.0700,140,15,0.0769
3,2024-01-04,184,10209.8200,55.4900,0.0700,146,14,0.0761
4,2024-01-05,189,10241.1400,54.1900,0.0700,134,15,0.0794


### 3.2 · order_items.csv  →  Daily Item-Level View (optional enrichment)

In [15]:
# Join items back to orders to get the order_date
items_with_date = order_items.merge(
    orders[['order_id', 'order_date']], on='order_id', how='left'
)

items_daily = (
    items_with_date
    .groupby('order_date')
    .agg(
        total_line_items = ('order_item_id', 'count'),
        total_qty_sold   = ('quantity',       'sum'),
        total_item_rev   = ('line_total_usd', 'sum'),
        avg_unit_price   = ('unit_price_usd', 'mean'),
    )
    .reset_index()
    .rename(columns={'order_date': 'date'})
)

print(f'items_daily shape: {items_daily.shape}')
print('Note: items_daily is an optional analytical view — not merged into master_dataset')
items_daily.head()

items_daily shape: (731, 5)
Note: items_daily is an optional analytical view — not merged into master_dataset


,date,total_line_items,total_qty_sold,total_item_rev,avg_unit_price
0,2024-01-01,461,930,10670.0000,13.8144
1,2024-01-02,470,966,11219.0200,14.2798
2,2024-01-03,495,984,10950.5600,13.6504
3,2024-01-04,471,931,10209.8200,13.9444
4,2024-01-05,481,941,10241.1400,12.9328


### 3.3 · marketing_spend_daily.csv  →  Daily Marketing Totals (5 channels → 1 row)

In [16]:
marketing_daily = (
    marketing
    .groupby('date')
    .agg(
        total_spend_usd              = ('spend_usd',              'sum'),
        total_impressions            = ('impressions',            'sum'),
        total_clicks                 = ('clicks',                 'sum'),
        total_conversions_marketing  = ('conversions',            'sum'),
        total_attributed_revenue_usd = ('attributed_revenue_usd', 'sum'),
    )
    .reset_index()
)

# ROAS = total attributed revenue / total spend  (NOT average of per-channel ROAS)
marketing_daily['avg_roas'] = (
    marketing_daily['total_attributed_revenue_usd'] / marketing_daily['total_spend_usd']
).round(3)

marketing_daily['total_spend_usd']              = marketing_daily['total_spend_usd'].round(2)
marketing_daily['total_attributed_revenue_usd'] = marketing_daily['total_attributed_revenue_usd'].round(2)

print(f'marketing_daily shape: {marketing_daily.shape}  (aggregated from {len(marketing):,} channel-day rows)')
marketing_daily.head()

marketing_daily shape: (731, 7)  (aggregated from 3,655 channel-day rows)


,date,total_spend_usd,total_impressions,total_clicks,total_conversions_marketing,total_attributed_revenue_usd,avg_roas
0,2024-01-01,12149.6400,2223769,84269,2479,486130.8500,40.0120
1,2024-01-02,12623.8900,2158313,52190,6109,1117332.3100,88.5090
2,2024-01-03,11628.2000,1864903,51258,5293,804760.5100,69.2080
3,2024-01-04,12705.5300,2091582,86234,5682,744334.0000,58.5830
4,2024-01-05,11597.8800,1496398,66045,2865,141325.9800,12.1860


### 3.4 · website_traffic_daily.csv  →  Already Daily (rename one column)

In [17]:
# Rename 'conversions' to 'conversions_web' to match master_dataset column naming convention
traffic_daily = traffic.rename(columns={'conversions': 'conversions_web'}).copy()

print(f'traffic_daily shape: {traffic_daily.shape}  (already one row per day — no aggregation needed)')
traffic_daily.head()

traffic_daily shape: (731, 8)  (already one row per day — no aggregation needed)


,date,sessions,unique_visitors,bounce_rate,pages_per_session,avg_session_duration_sec,conversion_rate,conversions_web
0,2024-01-01,7706,6486,0.4110,3.1900,162.9000,0.0250,192
1,2024-01-02,7759,6552,0.4110,3.3900,189.3000,0.0246,190
2,2024-01-03,7739,5958,0.4020,3.2900,183.5000,0.0245,189
3,2024-01-04,7723,5733,0.4740,2.9500,199.9000,0.0223,172
4,2024-01-05,7689,5882,0.4260,3.3900,170.6000,0.0243,186


### 3.5 · inventory_daily.csv  →  Daily Inventory Totals (500 products → 1 row)

In [18]:
inventory_daily = (
    inventory
    .groupby('date')
    .agg(
        total_stock_on_hand = ('stock_on_hand',    'sum'),
        total_units_sold    = ('units_sold',        'sum'),
        n_stockouts         = ('stockout_flag',     'sum'),
        n_reorders          = ('reorder_triggered', 'sum'),
    )
    .reset_index()
)

print(f'inventory_daily shape: {inventory_daily.shape}  (aggregated from {len(inventory):,} product-day rows)')
inventory_daily.head()

inventory_daily shape: (731, 5)  (aggregated from 365,500 product-day rows)


,date,total_stock_on_hand,total_units_sold,n_stockouts,n_reorders
0,2024-01-01,50000,326,0,1
1,2024-01-02,50045,275,0,2
2,2024-01-03,50490,307,0,1
3,2024-01-04,49534,356,0,0
4,2024-01-05,50214,419,0,0


### 3.6 · anomaly_log.csv  →  Add anomaly_flag = 1

In [19]:
anomaly_prepared = anomaly_log.copy()
anomaly_prepared['anomaly_flag'] = 1

print(f'anomaly_prepared shape: {anomaly_prepared.shape}  (will left-join on date — non-anomaly days get 0)')
anomaly_prepared.head()

anomaly_prepared shape: (20, 4)  (will left-join on date — non-anomaly days get 0)


,date,anomaly_event,anomaly_kpi,anomaly_flag
0,2024-02-08,economic_sentiment_drop,aov,1
1,2024-03-15,inventory_stockout,sales_volume,1
2,2024-04-22,logistics_disruption,fulfillment_time,1
3,2024-05-22,fraud_attack,refunds,1
4,2024-06-18,website_outage,sessions,1


---
## 4 · Merge All Tables Step-by-Step

Starting from `orders_daily` as the **spine** (every trading day has orders), we
left-join each prepared table on `date`.

```
orders_daily                              ← spine: 731 rows
    LEFT JOIN marketing_daily  ON date    ← adds 6 marketing cols  → 14 cols
    LEFT JOIN traffic_daily    ON date    ← adds 7 traffic cols    → 21 cols
    LEFT JOIN inventory_daily  ON date    ← adds 4 inventory cols  → 25 cols
    LEFT JOIN anomaly_prepared ON date    ← adds 3 label cols      → 28 cols
    LEFT JOIN latent_drivers   ON date    ← adds 5 driver cols     → 33 cols
```

### Merge 1 — Orders + Marketing

In [20]:
merged = orders_daily.merge(marketing_daily, on='date', how='left')

print(f'Shape after Merge 1: {merged.shape}  (orders_daily + marketing_daily)')
print(f'New columns added: {[c for c in merged.columns if c not in orders_daily.columns]}')
merged[['date', 'n_orders', 'total_revenue_usd', 'total_spend_usd', 'total_clicks', 'avg_roas']].head()

Shape after Merge 1: (731, 14)  (orders_daily + marketing_daily)
New columns added: ['total_spend_usd', 'total_impressions', 'total_clicks', 'total_conversions_marketing', 'total_attributed_revenue_usd', 'avg_roas']


,date,n_orders,total_revenue_usd,total_spend_usd,total_clicks,avg_roas
0,2024-01-01,194,10670.0000,12149.6400,84269,40.0120
1,2024-01-02,197,11219.0000,12623.8900,52190,88.5090
2,2024-01-03,195,10950.5400,11628.2000,51258,69.2080
3,2024-01-04,184,10209.8200,12705.5300,86234,58.5830
4,2024-01-05,189,10241.1400,11597.8800,66045,12.1860


### Merge 2 — + Website Traffic

In [21]:
merged = merged.merge(traffic_daily, on='date', how='left')

print(f'Shape after Merge 2: {merged.shape}  (+ traffic_daily)')
print(f'New columns added: {[c for c in traffic_daily.columns if c != "date"]}')
merged[['date', 'n_orders', 'total_revenue_usd', 'sessions', 'bounce_rate', 'conversion_rate', 'conversions_web']].head()

Shape after Merge 2: (731, 21)  (+ traffic_daily)
New columns added: ['sessions', 'unique_visitors', 'bounce_rate', 'pages_per_session', 'avg_session_duration_sec', 'conversion_rate', 'conversions_web']


,date,n_orders,total_revenue_usd,sessions,bounce_rate,conversion_rate,conversions_web
0,2024-01-01,194,10670.0000,7706,0.4110,0.0250,192
1,2024-01-02,197,11219.0000,7759,0.4110,0.0246,190
2,2024-01-03,195,10950.5400,7739,0.4020,0.0245,189
3,2024-01-04,184,10209.8200,7723,0.4740,0.0223,172
4,2024-01-05,189,10241.1400,7689,0.4260,0.0243,186


### Merge 3 — + Inventory

In [22]:
merged = merged.merge(inventory_daily, on='date', how='left')

print(f'Shape after Merge 3: {merged.shape}  (+ inventory_daily)')
print(f'New columns added: {[c for c in inventory_daily.columns if c != "date"]}')
merged[['date', 'n_orders', 'total_revenue_usd', 'sessions', 'total_stock_on_hand', 'total_units_sold', 'n_stockouts', 'n_reorders']].head()

Shape after Merge 3: (731, 25)  (+ inventory_daily)
New columns added: ['total_stock_on_hand', 'total_units_sold', 'n_stockouts', 'n_reorders']


,date,n_orders,total_revenue_usd,sessions,total_stock_on_hand,total_units_sold,n_stockouts,n_reorders
0,2024-01-01,194,10670.0000,7706,50000,326,0,1
1,2024-01-02,197,11219.0000,7759,50045,275,0,2
2,2024-01-03,195,10950.5400,7739,50490,307,0,1
3,2024-01-04,184,10209.8200,7723,49534,356,0,0
4,2024-01-05,189,10241.1400,7689,50214,419,0,0


### Merge 4 — + Anomaly Labels

In [23]:
merged = merged.merge(anomaly_prepared, on='date', how='left')

# Fill nulls: non-anomaly days get 0 flag and empty strings
merged['anomaly_flag']  = merged['anomaly_flag'].fillna(0).astype(int)
merged['anomaly_event'] = merged['anomaly_event'].fillna('')
merged['anomaly_kpi']   = merged['anomaly_kpi'].fillna('')

print(f'Shape after Merge 4: {merged.shape}  (+ anomaly_prepared)')
print(f'Anomaly days flagged: {merged["anomaly_flag"].sum()}')
merged[['date', 'n_orders', 'total_revenue_usd', 'sessions', 'anomaly_flag', 'anomaly_event', 'anomaly_kpi']].head(10)

Shape after Merge 4: (731, 28)  (+ anomaly_prepared)
Anomaly days flagged: 20


,date,n_orders,total_revenue_usd,sessions,anomaly_flag,anomaly_event,anomaly_kpi
0,2024-01-01,194,10670.0000,7706,0,,
1,2024-01-02,197,11219.0000,7759,0,,
2,2024-01-03,195,10950.5400,7739,0,,
3,2024-01-04,184,10209.8200,7723,0,,
4,2024-01-05,189,10241.1400,7689,0,,
5,2024-01-06,185,10221.0300,7719,0,,
6,2024-01-07,175,9183.9200,7647,0,,
7,2024-01-08,184,10055.4000,7706,0,,
8,2024-01-09,177,9513.0400,7684,0,,
9,2024-01-10,175,9479.1300,7697,0,,


### Merge 5 — + Latent Drivers (from master_dataset)

The five latent drivers are simulation-engine variables that do not originate from any
transactional table. They are stored in `master_dataset.csv` and joined in last.

In [24]:
master_ref = pd.read_csv(DATA / 'master_dataset.csv', parse_dates=['date'])

latent_drivers = master_ref[[
    'date', 'economic_index', 'marketing_pressure',
    'consumer_sentiment', 'seasonal_index', 'inventory_health'
]].copy()

merged = merged.merge(latent_drivers, on='date', how='left')

print(f'Shape after Merge 5: {merged.shape}  (+ latent_drivers)  ← FINAL')
print(f'All 33 columns: {list(merged.columns)}')
merged[['date', 'n_orders', 'total_revenue_usd', 'economic_index', 'marketing_pressure', 'consumer_sentiment', 'seasonal_index', 'inventory_health']].head()

Shape after Merge 5: (731, 33)  (+ latent_drivers)  ← FINAL


All 33 columns: ['date', 'n_orders', 'total_revenue_usd', 'avg_order_value_usd', 'avg_discount_pct', 'n_unique_customers', 'n_returns', 'return_rate', 'total_spend_usd', 'total_impressions', 'total_clicks', 'total_conversions_marketing', 'total_attributed_revenue_usd', 'avg_roas', 'sessions', 'unique_visitors', 'bounce_rate', 'pages_per_session', 'avg_session_duration_sec', 'conversion_rate', 'conversions_web', 'total_stock_on_hand', 'total_units_sold', 'n_stockouts', 'n_reorders', 'anomaly_event', 'anomaly_kpi', 'anomaly_flag', 'economic_index', 'marketing_pressure', 'consumer_sentiment', 'seasonal_index', 'inventory_health']


,date,n_orders,total_revenue_usd,economic_index,marketing_pressure,consumer_sentiment,seasonal_index,inventory_health
0,2024-01-01,194,10670.0000,0.0000,0.0000,0.0000,-0.1467,0.0000
1,2024-01-02,197,11219.0000,0.0650,-0.0292,0.0537,-0.1461,0.0030
2,2024-01-03,195,10950.5400,0.0386,-0.0302,0.0641,-0.1455,0.0327
3,2024-01-04,184,10209.8200,0.0163,-0.1767,0.0638,-0.1449,-0.0310
4,2024-01-05,189,10241.1400,-0.0271,-0.0442,0.0047,-0.1442,0.0143


---
## 5 · Final master_dataset — Reorder Columns

In [25]:
MASTER_COLUMN_ORDER = [
    'date',
    # Orders group
    'n_orders', 'total_revenue_usd', 'avg_order_value_usd',
    'avg_discount_pct', 'n_unique_customers', 'n_returns', 'return_rate',
    # Marketing group
    'total_spend_usd', 'total_impressions', 'total_clicks',
    'total_conversions_marketing', 'total_attributed_revenue_usd', 'avg_roas',
    # Traffic group
    'sessions', 'unique_visitors', 'bounce_rate', 'pages_per_session',
    'avg_session_duration_sec', 'conversion_rate', 'conversions_web',
    # Inventory group
    'total_stock_on_hand', 'total_units_sold', 'n_stockouts', 'n_reorders',
    # Latent drivers
    'economic_index', 'marketing_pressure', 'consumer_sentiment',
    'seasonal_index', 'inventory_health',
    # Anomaly labels
    'anomaly_flag', 'anomaly_event', 'anomaly_kpi',
]

master_rebuilt = merged[MASTER_COLUMN_ORDER].sort_values('date').reset_index(drop=True)

print(f'master_rebuilt shape: {master_rebuilt.shape}  (731 rows x 33 columns)')
master_rebuilt.head()

master_rebuilt shape: (731, 33)  (731 rows x 33 columns)


,date,n_orders,total_revenue_usd,avg_order_value_usd,avg_discount_pct,n_unique_customers,n_returns,return_rate,total_spend_usd,total_impressions,total_clicks,total_conversions_marketing,total_attributed_revenue_usd,avg_roas,sessions,unique_visitors,bounce_rate,pages_per_session,avg_session_duration_sec,conversion_rate,conversions_web,total_stock_on_hand,total_units_sold,n_stockouts,n_reorders,economic_index,marketing_pressure,consumer_sentiment,seasonal_index,inventory_health,anomaly_flag,anomaly_event,anomaly_kpi
0,2024-01-01,194,10670.0000,55.0000,0.0700,153,15,0.0773,12149.6400,2223769,84269,2479,486130.8500,40.0120,7706,6486,0.4110,3.1900,162.9000,0.0250,192,50000,326,0,1,0.0000,0.0000,0.0000,-0.1467,0.0000,0,,
1,2024-01-02,197,11219.0000,56.9500,0.0700,158,15,0.0761,12623.8900,2158313,52190,6109,1117332.3100,88.5090,7759,6552,0.4110,3.3900,189.3000,0.0246,190,50045,275,0,2,0.0650,-0.0292,0.0537,-0.1461,0.0030,0,,
2,2024-01-03,195,10950.5400,56.1600,0.0700,140,15,0.0769,11628.2000,1864903,51258,5293,804760.5100,69.2080,7739,5958,0.4020,3.2900,183.5000,0.0245,189,50490,307,0,1,0.0386,-0.0302,0.0641,-0.1455,0.0327,0,,
3,2024-01-04,184,10209.8200,55.4900,0.0700,146,14,0.0761,12705.5300,2091582,86234,5682,744334.0000,58.5830,7723,5733,0.4740,2.9500,199.9000,0.0223,172,49534,356,0,0,0.0163,-0.1767,0.0638,-0.1449,-0.0310,0,,
4,2024-01-05,189,10241.1400,54.1900,0.0700,134,15,0.0794,11597.8800,1496398,66045,2865,141325.9800,12.1860,7689,5882,0.4260,3.3900,170.6000,0.0243,186,50214,419,0,0,-0.0271,-0.0442,0.0047,-0.1442,0.0143,0,,


---
## 6 · Validate Against Original master_dataset.csv

### 6.1 · Shape, Date Range and Column Match

In [26]:
print('Original master_dataset.csv:')
print(f'  Shape  : {master_ref.shape}')
print(f'  Dates  : {master_ref["date"].min().date()} to {master_ref["date"].max().date()}')
print(f'  Cols   : {list(master_ref.columns)}')
print()
print('Rebuilt master_dataset:')
print(f'  Shape  : {master_rebuilt.shape}')
print(f'  Dates  : {master_rebuilt["date"].min().date()} to {master_rebuilt["date"].max().date()}')
print(f'  Cols   : {list(master_rebuilt.columns)}')

Original master_dataset.csv:
  Shape  : (731, 33)
  Dates  : 2024-01-01 to 2025-12-31
  Cols   : ['date', 'n_orders', 'total_revenue_usd', 'avg_order_value_usd', 'avg_discount_pct', 'n_unique_customers', 'n_returns', 'return_rate', 'total_spend_usd', 'total_impressions', 'total_clicks', 'total_conversions_marketing', 'total_attributed_revenue_usd', 'avg_roas', 'sessions', 'unique_visitors', 'bounce_rate', 'pages_per_session', 'avg_session_duration_sec', 'conversion_rate', 'conversions_web', 'total_stock_on_hand', 'total_units_sold', 'n_stockouts', 'n_reorders', 'economic_index', 'marketing_pressure', 'consumer_sentiment', 'seasonal_index', 'inventory_health', 'anomaly_flag', 'anomaly_event', 'anomaly_kpi']

Rebuilt master_dataset:
  Shape  : (731, 33)
  Dates  : 2024-01-01 to 2025-12-31
  Cols   : ['date', 'n_orders', 'total_revenue_usd', 'avg_order_value_usd', 'avg_discount_pct', 'n_unique_customers', 'n_returns', 'return_rate', 'total_spend_usd', 'total_impressions', 'total_clicks', 

### 6.2 · Traffic & Anomaly Labels (exact match expected — direct extraction)

In [27]:
exact_cols = ['sessions', 'unique_visitors', 'bounce_rate', 'pages_per_session',
              'avg_session_duration_sec', 'conversion_rate', 'conversions_web',
              'anomaly_flag', 'anomaly_event', 'anomaly_kpi']

results = []
for col in exact_cols:
    rb = master_rebuilt[col]
    rf = master_ref[col]
    if col in ('anomaly_event', 'anomaly_kpi'):
        # master_ref stores NaN on non-anomaly days; master_rebuilt uses ''
        match = (rb == rf.fillna('')).all()
    elif rb.dtype == object:
        match = (rb == rf).all()
    else:
        match = bool(np.allclose(rb.values.astype(float),
                                 rf.values.astype(float), rtol=1e-4))
    results.append({'column': col, 'exact_match': match})

pd.DataFrame(results)

,column,exact_match
0,sessions,True
1,unique_visitors,True
2,bounce_rate,True
3,pages_per_session,True
4,avg_session_duration_sec,True
5,conversion_rate,True
6,conversions_web,True
7,anomaly_flag,True
8,anomaly_event,True
9,anomaly_kpi,True


### 6.3 · All KPI Groups — Sum Comparison

In [28]:
numeric_cols = [
    'n_orders', 'total_revenue_usd', 'n_returns',
    'total_spend_usd', 'total_impressions', 'total_clicks', 'total_attributed_revenue_usd',
    'total_stock_on_hand', 'total_units_sold', 'n_stockouts', 'n_reorders'
]

summary = pd.DataFrame({
    'column':       numeric_cols,
    'original_sum': [master_ref[c].sum()      for c in numeric_cols],
    'rebuilt_sum':  [master_rebuilt[c].sum()  for c in numeric_cols],
})
summary['abs_diff'] = (summary['original_sum'] - summary['rebuilt_sum']).abs().round(2)
summary['pct_diff'] = (summary['abs_diff'] / summary['original_sum'] * 100).round(4)
summary

,column,original_sum,rebuilt_sum,abs_diff,pct_diff
0,n_orders,148084.0000,148084.0000,0.0000,0.0000
1,total_revenue_usd,8563983.4900,8563983.5600,0.0700,0.0000
2,n_returns,11441.0000,11441.0000,0.0000,0.0000
3,total_spend_usd,9347247.8400,9347247.8400,0.0000,0.0000
4,total_impressions,1306160062.0000,1306160062.0000,0.0000,0.0000
5,total_clicks,38292991.0000,38292991.0000,0.0000,0.0000
6,total_attributed_revenue_usd,318999435.7100,318999435.7100,0.0000,0.0000
7,total_stock_on_hand,36983959.0000,36983959.0000,0.0000,0.0000
8,total_units_sold,274688.0000,274688.0000,0.0000,0.0000
9,n_stockouts,44.0000,44.0000,0.0000,0.0000


### 6.4 · Side-by-Side Preview — Original vs Rebuilt

In [29]:
spot_cols = ['date', 'n_orders', 'total_revenue_usd', 'avg_roas',
             'sessions', 'n_stockouts', 'anomaly_flag', 'anomaly_event']

print('=== ORIGINAL master_dataset.csv (first 5 rows) ===')
display(master_ref[spot_cols].head())

print('=== REBUILT from 8 source tables (first 5 rows) ===')
display(master_rebuilt[spot_cols].head())

=== ORIGINAL master_dataset.csv (first 5 rows) ===


,date,n_orders,total_revenue_usd,avg_roas,sessions,n_stockouts,anomaly_flag,anomaly_event
0,2024-01-01,194,10670.0000,40.0120,7706,0,0,NaN
1,2024-01-02,197,11219.0000,88.5090,7759,0,0,NaN
2,2024-01-03,195,10950.5400,69.2080,7739,0,0,NaN
3,2024-01-04,184,10209.8200,58.5830,7723,0,0,NaN
4,2024-01-05,189,10241.1400,12.1860,7689,0,0,NaN


=== REBUILT from 8 source tables (first 5 rows) ===


,date,n_orders,total_revenue_usd,avg_roas,sessions,n_stockouts,anomaly_flag,anomaly_event
0,2024-01-01,194,10670.0000,40.0120,7706,0,0,
1,2024-01-02,197,11219.0000,88.5090,7759,0,0,
2,2024-01-03,195,10950.5400,69.2080,7739,0,0,
3,2024-01-04,184,10209.8200,58.5830,7723,0,0,
4,2024-01-05,189,10241.1400,12.1860,7689,0,0,


---
## 7 · Anomaly Days Spotlight

In [30]:
anomaly_days = master_rebuilt[master_rebuilt['anomaly_flag'] == 1][[
    'date', 'anomaly_event', 'anomaly_kpi',
    'total_revenue_usd', 'n_orders', 'avg_roas', 'sessions', 'n_stockouts'
]].reset_index(drop=True)

print(f'All {len(anomaly_days)} anomaly days in the rebuilt dataset:')
anomaly_days

All 20 anomaly days in the rebuilt dataset:


,date,anomaly_event,anomaly_kpi,total_revenue_usd,n_orders,avg_roas,sessions,n_stockouts
0,2024-02-08,economic_sentiment_drop,aov,8796.8600,177,26.3790,7664,0
1,2024-03-15,inventory_stockout,sales_volume,7563.4300,125,45.8360,8121,0
2,2024-04-22,logistics_disruption,fulfillment_time,12084.2400,212,34.2930,8536,0
3,2024-05-22,fraud_attack,refunds,11472.7500,203,70.7910,8305,0
4,2024-06-18,website_outage,sessions,14538.3700,241,33.8240,3053,0
5,2024-07-10,marketing_tracking_outage,roas,14540.2800,235,14.0320,9376,1
6,2024-08-05,defective_product_returns,return_rate,11670.9000,201,14.8680,8289,0
7,2024-08-20,back_to_school_surge,sales_volume,22652.0000,370,26.1370,9270,0
8,2024-09-03,email_campaign_spike,conversion_rate,10297.5400,188,23.8170,8073,0
9,2024-10-14,coupon_abuse,discount_rate,11595.9100,198,69.7980,7967,0


---
## 8 · Data Lineage Summary

```
SOURCE TABLE              GRAIN              JOIN KEY   AGGREGATION         MASTER COLUMNS
────────────────────────────────────────────────────────────────────────────────────────────────
customers.csv             per customer       customer_id  dimension join     (enrichment only)
products.csv              per product        product_id   dimension join     (enrichment only)

orders.csv                per order          order_date   groupby date    →  n_orders
                                                                             total_revenue_usd
                                                                             avg_order_value_usd
                                                                             avg_discount_pct
                                                                             n_unique_customers
                                                                             n_returns
                                                                             return_rate

marketing_spend_daily.csv per channel×day    date         groupby date    →  total_spend_usd
                                             (5→1 row)                       total_impressions
                                                                             total_clicks
                                                                             total_conversions_marketing
                                                                             total_attributed_revenue_usd
                                                                             avg_roas

website_traffic_daily.csv per day            date         rename column   →  sessions
                                                                             unique_visitors
                                                                             bounce_rate
                                                                             pages_per_session
                                                                             avg_session_duration_sec
                                                                             conversion_rate
                                                                             conversions_web

inventory_daily.csv       per product×day    date         groupby date    →  total_stock_on_hand
                                             (500→1 row)                     total_units_sold
                                                                             n_stockouts
                                                                             n_reorders

anomaly_log.csv           per event          date         left join       →  anomaly_flag
                                                          (fill 0/'')        anomaly_event
                                                                             anomaly_kpi

master_dataset.csv        per day            date         left join       →  economic_index
(latent drivers only)                        (sim vars)                      marketing_pressure
                                                                             consumer_sentiment
                                                                             seasonal_index
                                                                             inventory_health
────────────────────────────────────────────────────────────────────────────────────────────────
RESULT: master_dataset.csv   731 rows × 33 columns
```